In [1]:
import cv2
import numpy as np
import mediapipe as mp
import os
import pickle
from PIL import ImageFont, ImageDraw, Image

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [2]:
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

hands = mp_hands.Hands(
    static_image_mode=True,
    max_num_hands=1,
    min_detection_confidence=0.3
)

In [3]:
data = []
labels = []

DATA_DIR = './data/Train'

In [4]:
def extract_landmarks(results):
    all_landmarks = np.array(
        [[landmark.x, landmark.y, landmark.z] for hand_landmarks in results.multi_hand_landmarks for landmark in
         hand_landmarks.landmark]).flatten() if results.multi_hand_landmarks else np.zeros(21 * 3)
    return all_landmarks

In [5]:
image = cv2.imread('./data/MPTestRandomForest/1/2_132.png')
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
results = hands.process(image_rgb)
all_landmarks = extract_landmarks(results)

In [6]:
all_landmarks.shape

(63,)

In [17]:
#counter = 0
for curDir in os.listdir(DATA_DIR):
    for img_path in os.listdir(os.path.join(DATA_DIR, curDir)):
        if img_path.endswith('.png'):
            data_aux = []
            image = cv2.imread(os.path.join(DATA_DIR, curDir, img_path))
            image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
            #if counter % 2 == 0:
            #   flipped_image = cv2.flip(image_rgb, 1)
            #    cv2.imwrite(os.path.join(DATA_DIR, curDir, img_path), cv2.cvtColor(flipped_image, cv2.COLOR_BGR2RGB))
    
            #counter += 1
    
            results = hands.process(image_rgb)
            if results.multi_hand_landmarks:
                for hand_landmarks in results.multi_hand_landmarks:
                    for landmark in hand_landmarks.landmark:
                        x, y, z = landmark.x, landmark.y, landmark.z
                        data_aux.append(x)
                        data_aux.append(y)
                        data_aux.append(z)
    
                data.append(data_aux)
                labels.append(curDir)

f = open('data.pickle', 'wb')
pickle.dump({'data': data, 'labels': labels}, f)
f.close()

In [12]:
data_dict = pickle.load(open('data.pickle', 'rb'))

data = np.asarray(data_dict['data'])
labels = np.asarray(data_dict['labels'])

x_train, x_test, y_train, y_test = train_test_split(data, labels, test_size=0.2, random_state=42, stratify=labels)

clf = RandomForestClassifier(n_estimators=100, random_state=42)

clf.fit(x_train, y_train)

y_predict = clf.predict(x_test)

print(accuracy_score(y_test, y_predict))

1.0


In [8]:
def draw_landmarks(image, results):
    mp_drawing.draw_landmarks(image, results.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS,
                              mp_drawing.DrawingSpec(color=(0, 32, 128), thickness=2, circle_radius=2),
                              mp_drawing.DrawingSpec(color=(255, 255, 255), thickness=2))

In [13]:
cap = cv2.VideoCapture('./data/Test/1_7.mp4')

labels_dict = {}
with open('./data/rda/label_map.txt', 'r', encoding='utf-8') as f:
    for idx, line in enumerate(f, 1):
        labels_dict[idx] = line.strip()
        
while True:
    data_aux = []
    x_arr = []
    y_arr = []

    ret, frame = cap.read()
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    results = hands.process(frame_rgb)
    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            for landmark in hand_landmarks.landmark:
                x, y, z = landmark.x, landmark.y, landmark.z
                data_aux.append(x)
                data_aux.append(y)
                data_aux.append(z)
                x_arr.append(x)
                y_arr.append(y)

        draw_landmarks(frame, results)

        x1 = int(min(x_arr) * frame.shape[1] - 10)
        x2 = int(max(x_arr) * frame.shape[1] + 10)
        y1 = int(min(y_arr) * frame.shape[0] - 10)
        y2 = int(max(y_arr) * frame.shape[0] + 10)

        prediction = clf.predict([data_aux])

        prediction_character = labels_dict[int(prediction[0])]

        frame_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        font_path = "./fonts/arial.ttf"
        font = ImageFont.truetype(font_path, 32)

        draw = ImageDraw.Draw(frame_pil)
        draw.text((x1, y1 + 10), prediction_character, font=font, fill=(0, 255, 0))

        frame = cv2.cvtColor(np.array(frame_pil), cv2.COLOR_RGB2BGR)

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 4)

    cv2.imshow('Hand Tracking', frame)
    key = cv2.waitKey(1) & 0xFF

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
